# Queries in Weaviate

Weaviate offers three main search approaches, each with different strengths:

- **Keyword search (BM25)**: Precise, exact matches
- **Vector search**: Semantic understanding, handles typos
- **Hybrid search**: Combines both approaches

Let's try them out here.

In [ ]:
# Install weaviate-client...

## TASK 1
import weaviate
from openai import OpenAI as OpenAIClient

raw_client = OpenAIClient()
API_KEY  = str(raw_client.api_key)
API_BASE = str(raw_client.base_url)

HEADERS = {
    "X-OpenAI-Api-Key": API_KEY,
    "X-OpenAI-BaseURL": API_BASE.rstrip("/").removesuffix("/v1"),
}

try:
    client = weaviate.connect_to_embedded(
        version="1.32.3",
        headers=HEADERS,
        environment_variables={"LOG_LEVEL": "error"},
    )
except weaviate.exceptions.WeaviateStartUpError:
    # If already running, connect to it instead
    client = weaviate.connect_to_local(
        port=8079,
        grpc_port=50050,
        headers=HEADERS,
    )

movies = client.collections.get("Movies")

response = movies.query.bm25(
    query="space",
    limit=5,
    query_properties=["title"]
)

print(len(response.objects), "objects found")
for o in response.objects:
    print(f"Title: {o.properties['title']}")
    print(f"Release Year: {o.properties['release_year']}")
    print()

response = movies.query.bm25(
    query="spaace",
    limit=5,
    query_properties=["title"]
)

print(len(response.objects), "objects found")
for o in response.objects:
    print(f"Title: {o.properties['title']}")
    print(f"Release Year: {o.properties['release_year']}")
    print()

## TASK 2
response = movies.query.near_text(
    query="space",
    limit=3,
)

print(len(response.objects), "objects found")
for o in response.objects:
    print(f"Title: {o.properties['title']}")
    print(f"Release Year: {o.properties['release_year']}")
    print()
    
response = movies.query.near_text(
    query="spaace",
    limit=3,
)

print(len(response.objects), "objects found")
for o in response.objects:
    print(f"Title: {o.properties['title']}")
    print(f"Release Year: {o.properties['release_year']}")
    print()
    
## TASK 3
response = movies.query.hybrid(
    query="space",
    limit=3,
    alpha=0.5  # Balance between keyword & vector search
)

print(len(response.objects), "objects found")
for o in response.objects:
    print(f"Title: {o.properties['title']}")
    print(f"Release Year: {o.properties['release_year']}")
    print()

## TASK 4
from weaviate.classes.query import Filter

response = movies.query.near_text(
    query="science fiction",
    limit=3,
    filters=Filter.by_property("release_year").greater_than(2015)
)

print(len(response.objects), "objects found")
for o in response.objects:
    print(f"Title: {o.properties['title']}")
    print(f"Release Year: {o.properties['release_year']}")
    print()

response = movies.query.near_text(
    query="science fiction",
    limit=3,
    filters=(
        Filter.by_property("release_year").less_than(2015) &
        Filter.by_property("popularity").greater_than(100)
    )
)

print(len(response.objects), "objects found")
for o in response.objects:
    print(f"Title: {o.properties['title']}")
    print(f"Release Year: {o.properties['release_year']}")
    print()
    
client.close()


**Run these cells to prepare the `movies` client once again.**

In [1]:
!pip install weaviate-client==4.16.10 pydantic==2.11.9

In [2]:
import weaviate
from openai import OpenAI as OpenAIClient

raw_client = OpenAIClient()
API_KEY  = str(raw_client.api_key)
API_BASE = str(raw_client.base_url)

HEADERS = {
    "X-OpenAI-Api-Key": API_KEY,
    "X-OpenAI-BaseURL": API_BASE.rstrip("/").removesuffix("/v1"),
}

try:
    client = weaviate.connect_to_embedded(
        version="1.32.3",
        headers=HEADERS,
        environment_variables={"LOG_LEVEL": "error"},
    )
except weaviate.exceptions.WeaviateStartUpError:
    # If already running, connect to it instead
    client = weaviate.connect_to_local(
        port=8079,
        grpc_port=50050,
        headers=HEADERS,
    )

⚠️ **Run the hidden cell to re-create the `movies` collection.** ⚠️

In [3]:
from datasets import load_dataset

movies_dataset = load_dataset("weaviate/agents", "personalization-agent-movies", split="train")
movies_data = movies_dataset[:200]["properties"]

from weaviate.classes.config import Property, DataType, Configure

# Delete "Movies" if it's already present to allow repeated runs
if "Movies" in client.collections.list_all():
    client.collections.delete("Movies")

client.collections.create(
    name="Movies",
    properties=[
        Property(
            name="title",
            data_type=DataType.TEXT,
        ),
        Property(
            name="overview",
            data_type=DataType.TEXT,
        ),
        Property(
            name="release_year",
            data_type=DataType.INT,
        ),
        Property(
            name="popularity",
            data_type=DataType.NUMBER,
        ),
    ],
    vector_config=[
        Configure.Vectors.text2vec_openai(
            name="default",
            source_properties=["title", "overview"],
            model="text-embedding-3-small"
        )
    ]
)

movies = client.collections.use("Movies")

def preprocess_row(item: dict) -> dict:
    obj = {
        "title": item["title"],
        "overview": item["overview"],
        "release_year": item["release_date"].year,
        "popularity": item["popularity"],
    }
    return obj

from weaviate.util import generate_uuid5

with movies.batch.fixed_size(batch_size=100) as batch:
    for item in movies_data:
        obj = preprocess_row(item)

        # Add object to batch for import
        batch.add_object(
            properties=obj,
            uuid=generate_uuid5(item["title"]+item["overview"])
        )

### Keyword search

First, let's try a keyword search, which scores results using exact matches.

![images/search_keyword.png](images/search_keyword.png)

**Perform a keyword search using BM25 on first the query `"space"` and then `"spaace"`.**

In [4]:
response = movies.query.____(
    query="____",
    query_properties=["title"],
    limit=5
)

print(len(response.objects), "objects found")
for o in response.objects:
    print(f"Title: {o.properties['title']}")
    print(f"Release Year: {o.properties['release_year']}")
    print()

Keyword search excels at finding exact text matches. It's fast and efficient for precise queries, but struggles with typos, synonyms, or semantic variations.

In [5]:
response = ____(
    query="____",
    query_properties=["title"],
    limit=5
)

print(len(response.objects), "objects found")
for o in response.objects:
    print(f"Title: {o.properties['title']}")
    print(f"Release Year: {o.properties['release_year']}")
    print()

### Vector search

Vector search uses embeddings to understand meaning rather than exact text matches. This makes it more robust for natural language queries and handles typos gracefully.

![images/search_vector.png](images/search_vector.png)

**Perform a vector search on the same two queries and compare the results.**

In [6]:
response = movies.____.____(
    query="space",
    limit=3,
)

print(len(response.objects), "objects found")
for o in response.objects:
    print(f"Title: {o.properties['title']}")
    print(f"Release Year: {o.properties['release_year']}")
    print()

Notice that it's picking up results without the exact keyword match. That's because it's based on meaning! How well does vector search work given typos or synonyms?

In [7]:
response = movies.query.near_text(
    query="spaace",
    limit=3,
)

print(len(response.objects), "objects found")
for o in response.objects:
    print(f"Title: {o.properties['title']}")
    print(f"Release Year: {o.properties['release_year']}")
    print()

Quite well, as it turns out. The embedding models do a good job at estimating what the words might mean, even if it includes typos.

Vector search's semantic understanding allows it to find relevant results even with typos or synonyms. This makes it ideal for natural language queries where users might not use exact keywords.

### Hybrid search 

Lastly, let's take a look at hybrid search.

Hybrid search combines the precision of keyword search with the semantic understanding of vector search. The `alpha` parameter controls the balance between the two approaches (`0.0` = pure keyword search, `1.0` = pure vector search).

![images/search_hybrid.png](images/search_hybrid.png)

**Perform a hybrid search on the query `"space"` with an alpha value of `0.5` and compare to the keyword and vector search results.**

In [8]:
response = movies.query.____(
    query="space",
    ____,  # Balance between keyword & vector search
    limit=3
)

print(len(response.objects), "objects found")
for o in response.objects:
    print(f"Title: {o.properties['title']}")
    print(f"Release Year: {o.properties['release_year']}")
    print()

Hybrid search is particularly powerful in production systems where you want both precision and semantic understanding. It often provides the best overall search experience.

### Filters

Filters allow you to narrow search results based on metadata properties. They can be combined with any search type and use Boolean operators for complex conditions.

**Perform a vector search using the query `"science fiction"` and applying a filter for `release_year` **greater than** `2015`.**

In [9]:
from weaviate.classes.query import Filter

response = movies.query.near_text(
    query="____",
    limit=3,
    filters=Filter.____("____").____(____)
)

print(len(response.objects), "objects found")
for o in response.objects:
    print(f"Title: {o.properties['title']}")
    print(f"Release Year: {o.properties['release_year']}")
    print()

Filters work with any search type and can significantly improve search relevance by focusing on specific criteria like date ranges, categories, or other texts.

Any number of filters can be combined in Weaviate using Boolean operators.

**Perform the same query again, but using two filters: `release_year` **less than** `2015` and `popularity` **greater than** `100`.**

In [10]:
from weaviate.classes.query import Filter

response = movies.query.near_text(
    query="science fiction",
    limit=3,
    filters=(
        Filter.____("____").____(____) &
        Filter.____("____").____(____)
    )
)

print(len(response.objects), "objects found")
for o in response.objects:
    print(f"Title: {o.properties['title']}")
    print(f"Release Year: {o.properties['release_year']}")
    print()

### Close the client

Remember to close the client library connection & Weaviate.

In [11]:
client.close()